In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

In [4]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from datasets import load_dataset
import tqdm

tokenizer = AutoTokenizer.from_pretrained("TehranNLP-org/roberta-base-mrpc-2e-5-42")
model = AutoModelForSequenceClassification.from_pretrained("TehranNLP-org/roberta-base-mrpc-2e-5-42")

tokenizer_config.json:   0%|          | 0.00/327 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/415 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

In [5]:
print(model)

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [6]:
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device='cuda')

In [7]:
# two examples from training set

sentence = "Amrozi accused his brother , whom he called \" the witness \" , of deliberately distorting his evidence . Referring to him as only \" the witness \" , Amrozi accused his brother of deliberately distorting his evidence ." # 1 equaivalent

print(classifier(sentence))

sentence = "Yucaipa owned Dominick 's before selling the chain to Safeway in 1998 for $ 2.5 billion . Yucaipa bought Dominick 's in 1995 for $ 693 million and sold it to Safeway for $ 1.8 billion in 1998 ." # 0 not_equivalent

print(classifier(sentence))

[{'label': 'LABEL_1', 'score': 0.9977443218231201}]
[{'label': 'LABEL_0', 'score': 0.9884207248687744}]


In [8]:
dataset = load_dataset("glue", "mrpc")['validation']
print(dataset)

Generating train split:   0%|          | 0/3668 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/408 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1725 [00:00<?, ? examples/s]

Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})


In [11]:
correct = 0
total = 0

for i in tqdm.tqdm(range(408)):
    result = classifier(dataset[i]['sentence1'] + ' ' + dataset[i]['sentence2'])

    if result[0]['label'] == 'LABEL_1':
        result = 1
    elif result[0]['label'] == 'LABEL_0':
        result = 0
    else:
        exit()
        
    if result == dataset[i]['label']:
        correct += 1
    total += 1

print("correct:{}, total:{}, accuracy:{}".format(correct, total, correct/total))

100%|██████████| 408/408 [00:02<00:00, 166.17it/s]

correct:359, total:408, accuracy:0.8799019607843137
